# FedDerm: GPU DP-LoRA Privacy-Utility Sweep

This notebook executes the federated **DP-LoRA (Vision Transformer ViT-B/16 + LoRA adapters $r=8$)** privacy-utility experiments for **FedDerm** on a free cloud GPU runtime (**Google Colab** or **Kaggle Notebooks**).

---

## Instructions for Google Colab
1. **Enable GPU Accelerator**:
   - Navigate to `Runtime` -> `Change runtime type`.
   - Select **T4 GPU** (or any available GPU accelerator) and click `Save`.
2. **Provide Repository Code**:
   - **Option A (Git Clone)**: Provide your GitHub repo URL in the setup cell below.
   - **Option B (Zip Upload)**: Upload `FedDerm.zip` via the Colab files panel on the left and unzip it.
3. **Run Experiments**:
   - Click `Runtime` -> `Run all`.
   - Expected total execution time: **~1 to 1.5 hours** for the complete multi-seed sweep.
   - The final cell will automatically prompt a browser download of `dp_lora_fedprox_results.zip`.

---

## Instructions for Kaggle Notebooks (Recommended for Unattended Runs)
1. **Enable GPU Accelerator**:
   - In the right-hand **Notebook settings** panel, under **Accelerator**, select **GPU T4 x2** or **GPU P100**.
   - Under **Internet**, toggle the switch to **On** (required for pip packages and downloading pretrained ViT weights).
2. **Provide Repository Code**:
   - **Option A (Git Clone)**: Enter your GitHub repo URL in the setup cell below.
   - **Option B (Kaggle Dataset)**: Add the uploaded repository as a dataset (mounted under `/kaggle/input/`).
3. **Execute via Background Commit**:
   - Click **Save Version** -> Select **Save & Run All (Commit)** -> Click **Save**.
   - You can safely close your browser tab. The notebook will run top-to-bottom on Kaggle cloud infrastructure.
   - Once completed (~1-1.5h), navigate to the **Output** tab of the notebook version and download `dp_lora_fedprox_results.zip`.


In [ ]:
# Cell 1: Platform Detection
import os
import sys
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/working")

if IS_COLAB:
    print("[Environment] Detected Google Colab Runtime")
    BASE_DIR = Path("/content")
elif IS_KAGGLE:
    print("[Environment] Detected Kaggle Notebook Runtime")
    BASE_DIR = Path("/kaggle/working")
else:
    print("[Environment] Detected Local / Generic Python Environment")
    BASE_DIR = Path.cwd()

print(f"[Environment] Base working directory: {BASE_DIR}")


In [ ]:
# Cell 2: Repository Setup & Package Installation
import subprocess
import shutil

REPO_GITHUB_URL = "https://github.com/yusufcalisir/FedDerm.git"  # Update if using a private fork
REPO_DIR_NAME = "FedDerm"

repo_path = BASE_DIR / REPO_DIR_NAME

if not (repo_path / "pyproject.toml").exists():
    # Check if repo was uploaded as a Kaggle dataset under /kaggle/input
    kaggle_input_dirs = list(Path("/kaggle/input").glob("*")) if Path("/kaggle/input").exists() else []
    found_dataset = False
    for d in kaggle_input_dirs:
        if (d / "pyproject.toml").exists() or (d / REPO_DIR_NAME / "pyproject.toml").exists():
            src_dir = d if (d / "pyproject.toml").exists() else (d / REPO_DIR_NAME)
            print(f"[Setup] Found repository in Kaggle input dataset: {src_dir}")
            shutil.copytree(src_dir, repo_path, dirs_exist_ok=True)
            found_dataset = True
            break
    
    if not found_dataset:
        print(f"[Setup] Cloning repository from {REPO_GITHUB_URL}...")
        subprocess.run(["git", "clone", REPO_GITHUB_URL, str(repo_path)], check=True)

os.chdir(str(repo_path))
print(f"[Setup] Current working directory: {Path.cwd()}")

# Install local package and dependencies in editable mode
print("[Setup] Installing FedDerm with PEFT/ViT dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[peft]"], check=True)
print("[Setup] Package installation completed successfully.")


In [ ]:
# Cell 3: GPU Acceleration Verification
import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if not cuda_available:
    error_msg = (
        "FATAL ERROR: No GPU accelerator detected!\n\n"
        "- On Google Colab: Go to Runtime -> Change runtime type -> Select T4 GPU -> Save.\n"
        "- On Kaggle: In the right sidebar under Notebook settings -> Accelerator -> Select GPU T4 x2 or P100.\n"
    )
    raise RuntimeError(error_msg)

device_count = torch.cuda.device_count()
device_name = torch.cuda.get_device_name(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print(f"Detected GPU: {device_name} (Count: {device_count})")
print(f"Total GPU VRAM: {total_vram_gb:.2f} GB")


In [ ]:
# Cell 4: Single-Batch GPU Benchmark vs CPU Baseline
import time
import torch.nn as nn
from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from fedderm.models import build_vit_lora_model

print("=" * 70)
print("SINGLE-BATCH TIMING BENCHMARK (GPU vs. CPU)")
print("=" * 70)

device = torch.device("cuda")
batch_size = 64

# Load model
print("Loading ViT-B/16 with LoRA (r=8) onto GPU...")
t0 = time.time()
model = build_vit_lora_model("vit_base_patch16_224", num_classes=7, rank=8, pretrained=True).to(device)
print(f"Model loaded in {time.time() - t0:.2f}s")

# Check Opacus compatibility
errors = ModuleValidator.validate(model, strict=False)
assert len(errors) == 0, f"Opacus validation failed: {errors}"
print("Opacus ModuleValidator: PASSED (100% compatible)")

# Dummy batch (224x224)
dummy_x = torch.randn(batch_size, 3, 224, 224, device=device)
dummy_y = torch.randint(0, 7, (batch_size,), device=device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

# Non-DP Timing
for _ in range(3):  # Warmup
    optimizer.zero_grad()
    loss = criterion(model(dummy_x), dummy_y)
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()

t0 = time.time()
n_iters = 20
for _ in range(n_iters):
    optimizer.zero_grad()
    loss = criterion(model(dummy_x), dummy_y)
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()
gpu_non_dp_batch_s = (time.time() - t0) / n_iters

# DP-SGD Timing
privacy_engine = PrivacyEngine(secure_mode=False)
dp_model, dp_opt, dp_loader = privacy_engine.make_private(
    module=model,
    optimizer=optimizer,
    data_loader=torch.utils.data.DataLoader(torch.utils.data.TensorDataset(dummy_x, dummy_y), batch_size=batch_size),
    noise_multiplier=1.0,
    max_grad_norm=1.0,
)

for x, y in dp_loader:
    dp_opt.zero_grad()
    loss = criterion(dp_model(x), y)
    loss.backward()
    dp_opt.step()
    break
torch.cuda.synchronize()

t0 = time.time()
for _ in range(n_iters):
    for x, y in dp_loader:
        dp_opt.zero_grad()
        loss = criterion(dp_model(x), y)
        loss.backward()
        dp_opt.step()
torch.cuda.synchronize()
gpu_dp_batch_s = (time.time() - t0) / n_iters

print(f"\n[Benchmark Results (Batch Size 64)]:")
print(f"  Non-DP Pass: GPU = {gpu_non_dp_batch_s*1000:.1f} ms/batch | CPU = 46,914 ms/batch | Speedup = {46.914 / gpu_non_dp_batch_s:.1f}x")
print(f"  DP-SGD Pass: GPU = {gpu_dp_batch_s*1000:.1f} ms/batch | CPU = 50,255 ms/batch | Speedup = {50.255 / gpu_dp_batch_s:.1f}x")
print("=" * 70)


In [ ]:
# Cell 5: Stage 1 -- Non-DP Sanity Check Run (1 Run)
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))
from scripts.run_dp_lora_sweep import run_sweep

print("=" * 70)
print("RUNNING STAGE 1: NON-DP SANITY CHECK")
print("=" * 70)

run_sweep(run_sanity=True, run_dp=False, results_base="results/dp_lora_fedprox")


In [ ]:
# Cell 6: Stage 2 -- Full Multi-Seed DP Sweep (4 Noise Levels x 3 Seeds = 12 Runs)
print("=" * 70)
print("RUNNING STAGE 2: MULTI-SEED DP-SGD SWEEP")
print("=" * 70)

NOISE_MULTIPLIERS = [0.3, 0.5, 1.0, 2.0]
SEEDS = [42, 43, 44]

run_sweep(
    noise_multipliers=NOISE_MULTIPLIERS,
    seeds=SEEDS,
    run_sanity=False,
    run_dp=True,
    results_base="results/dp_lora_fedprox",
)


In [ ]:
# Cell 7: Display Sweep Summary Results & Tradeoff Plot
import pandas as pd
from IPython.display import Image, display

summary_csv_path = Path("results/dp_lora_fedprox/multiseed_summary.csv")
plot_path = Path("results/dp_lora_fedprox/privacy_utility_tradeoff_multiseed.png")

if summary_csv_path.exists():
    df = pd.read_csv(summary_csv_path)
    print("\n" + "=" * 75)
    print("DP-LoRA FEDPROX MULTI-SEED SWEEP SUMMARY TABLE")
    print("=" * 75)
    display(df)

if plot_path.exists():
    print("\n[Privacy-Utility Tradeoff Plot]")
    display(Image(filename=str(plot_path)))


In [ ]:
# Cell 8: Package & Export Results Archive
import zipfile
from pathlib import Path

results_src = Path("results/dp_lora_fedprox")
zip_filename = "dp_lora_fedprox_results.zip"

if IS_KAGGLE:
    out_zip_path = Path("/kaggle/working") / zip_filename
else:
    out_zip_path = Path.cwd() / zip_filename

print(f"Archiving results from {results_src} into {out_zip_path}...")
with zipfile.ZipFile(out_zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in results_src.rglob("*"):
        if file_path.is_file():
            arcname = file_path.relative_to(results_src.parent)
            zipf.write(file_path, arcname=arcname)

print(f"Results successfully archived: {out_zip_path} ({out_zip_path.stat().st_size / (1024*1024):.2f} MB)")

if IS_COLAB:
    from google.colab import files
    print("Triggering browser download for Colab...")
    files.download(str(out_zip_path))
elif IS_KAGGLE:
    print("\n" + "=" * 75)
    print("KAGGLE UNATTENDED RUN COMPLETE!")
    print("=" * 75)
    print(f"The zip archive '{zip_filename}' is saved in /kaggle/working/.")
    print("You can download it from the 'Output' section of this notebook version.")
    print("To reintegrate locally:")
    print("  1. Download dp_lora_fedprox_results.zip")
    print("  2. Extract contents into the FedDerm 'results/dp_lora_fedprox/' folder.")
    print("=" * 75)
